# laya-cuda-bench — report

Every table and chart below is computed from `results/` only. Cost assumptions are the constants in the next cell; change them and re-run.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
import pandas as pd, matplotlib.pyplot as plt
from harness import report as R

# ---- assumptions (stated in the post) ----
R.USD_PER_KWH = 0.12          # $/kWh
R.LIFETIME_YEARS = 3
R.CARD_USD.update({})         # override street prices here if needed
UTILISATION = [1.0, 0.4]      # 24/7 at the sustained rate, and a 40%-utilised fleet
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

## 1. Parity gate (per box)

In [ ]:
rows = []
for p in R.RESULTS.glob('*/parity.json'):
    d = json.loads(p.read_text())
    for r in d['results']:
        rows.append({'box': p.parent.name, 'gpu': d['gpu'], 'model': r['model'], 'backend': r['backend'],
                     'argmax': f"{r['argmax_agreements']}/{r['questions']}", 'max_prob_err': r['probability_max_abs_error'],
                     'alloc_growth_B': r['stability']['allocated_growth_bytes'], 'verdict': r['verdict']})
parity = pd.DataFrame(rows)
parity.sort_values(['box', 'model', 'backend'])

## 2. Backend shootout — latency vs batch (median of per-repeat medians, min/max spread)

In [ ]:
cells = R.shootout()
if not cells.empty:
    for (box, model), g in cells.groupby(['box', 'model']):
        fig, ax = plt.subplots(figsize=(7, 4))
        for (backend, length), gg in g.groupby(['backend', 'length']):
            gg = gg.sort_values('batch')
            ax.errorbar(gg['batch'], gg['median_ms'], yerr=[gg['median_ms'] - gg['min_ms'], gg['max_ms'] - gg['median_ms']],
                        marker='o', capsize=2, label=f'{backend} / {length}', ls='-' if length == 'short' else '--')
        ax.set_xscale('log', base=2); ax.set_yscale('log'); ax.set_xlabel('batch (rows per forward)'); ax.set_ylabel('latency ms')
        ax.set_title(f'{box} — {model} — {g["gpu"].iloc[0]}'); ax.grid(True, which='both', alpha=.3); ax.legend(fontsize=7)
    display(cells.pivot_table(index=['box', 'model', 'backend', 'length'], columns='batch', values='median_ms').round(2))
else:
    print('no shootout results yet')

## 3. Server scenario — latency vs offered load, and sustained decisions/s under each SLO

In [ ]:
steps, sus = R.sweep_steps(), R.sweeps()
if not steps.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for key, g in steps.groupby(['box', 'model', 'label']):
        g = g.sort_values('target_qps')
        ax.plot(g['decisions_per_s'], g['p99_ms'], marker='o', label=' / '.join(k for k in key if k))
    for slo in (50, 130):
        ax.axhline(slo, color='grey', ls=':', lw=1); ax.text(ax.get_xlim()[0], slo, f' p99 = {slo} ms', va='bottom', fontsize=8)
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('achieved decisions / s'); ax.set_ylabel('p99 request latency (ms)')
    ax.set_title('Server scenario: tail latency vs offered load'); ax.grid(True, which='both', alpha=.3); ax.legend(fontsize=7)
    sus['decisions_per_day_M'] = sus['decisions_per_s'] * 86400 / 1e6
    display(sus.pivot_table(index=['box', 'model', 'label', 'backend'], columns='slo_ms', values=['decisions_per_s', 'decisions_per_day_M', 'watts_mean']).round(1))
else:
    print('no sweeps yet')

## 4. Offline ceiling and baselines

In [ ]:
off = R.offline(); base = R.baselines(); jev = R.jev()
if not off.empty: display(off[['box', 'gpu', 'model', 'backend', 'length', 'batch', 'decisions_per_s', 'watts_mean', 'joules_per_decision', 'peak_vram_mb']].round(3))
if not base.empty: display(base[['box', 'label', 'concurrency', 'ok', 'p50_ms', 'p99_ms', 'decisions_per_s', 'mean_prompt_tokens', 'mean_completion_tokens']].round(1))
if not jev.empty: display(jev[['concurrency', 'ok', 'p50_ms', 'p95_ms', 'p99_ms', 'achieved_qps', 'mean_input_tokens_per_request', 'rate_limited']].round(1))

## 5. Cost per million decisions (at the p99 ≤ 130 ms operating point)

In [ ]:
rows = []
if not sus.empty:
    gpu_of = {p.parent.name: json.loads(p.read_text())['gpu'] for p in R.RESULTS.glob('*/parity.json')}
    for _, s in sus[(sus.slo_ms == 130) & sus.decisions_per_s.notna()].iterrows():
        for u in UTILISATION:
            c = R.cost_per_million(s['decisions_per_s'], s['watts_mean'] or 0, gpu_of.get(s['box'], ''), utilisation=u)
            rows.append({'column': f"{s['box']} {s['label'] or 'whole'} {s['model']}", 'utilisation': u, **c})
if not jev.empty:
    tok = jev['mean_input_tokens_per_request'].mean()
    rows.append({'column': 'Jev API (list price)', 'utilisation': None, 'energy_usd_per_m': 0, 'amortised_usd_per_m': 0,
                 'total_usd_per_m': R.jev_cost_per_million(tok)})
cost = pd.DataFrame(rows)
if not cost.empty:
    display(cost.round(3))
    fig, ax = plt.subplots(figsize=(8, 4))
    c1 = cost[(cost.utilisation == 1.0) | cost.utilisation.isna()]
    ax.bar(c1['column'], c1['energy_usd_per_m'], label='energy'); ax.bar(c1['column'], c1['amortised_usd_per_m'], bottom=c1['energy_usd_per_m'], label='amortised hardware (3 yr, 24/7)')
    ax.set_ylabel('$ per million decisions'); ax.set_yscale('log'); ax.legend(); plt.xticks(rotation=30, ha='right'); ax.grid(True, axis='y', alpha=.3)